# Ethiopia Food Prices — 10 SPIKE CLASSIFICATION

Train a supervised model to classify the next observation as a spike or non-spike.


In [ ]:
# The feature table from the previous step contains engineered signals such as lagged prices,
# rolling trends, volatility, and relative price levels. These variables are useful for predicting
# whether the next observation is an abnormal spike.
#
# We convert the spike flag into a binary target and train models to classify each record as
# either spike (1) or non-spike (0). This is a supervised classification problem.
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import joblib

root = Path('../')
feature_df = pd.read_csv(root / 'data' / 'eth_food_prices_features.csv', parse_dates=['date'])
feature_df = feature_df.dropna().copy()
feature_df['is_spike_target'] = feature_df['is_spike_flag'].astype(int)
feature_df.head()


,date,market,commodity,commodity_unit,admin1,year,month_of_year,usdprice,lag_1,lag_3,...,rolling_mean_12,rolling_std_3,rolling_std_6,rolling_std_12,pct_change_1m,price_to_mean_ratio,price_outlier_flag,is_spike_flag,is_drop,is_spike_target
0,2026-07-15,Abaala,Butter (goat milk),Butter (goat milk) (KG),Afar,2026,7,9.35,9.38,9.53,...,8.905000,0.079373,1.538555,1.538555,-1.263158,1.042529,False,0,False,0
1,2026-06-15,Abaala,Garlic,Garlic (KG),Afar,2026,6,2.38,2.53,2.56,...,2.855000,0.015275,0.742341,0.742341,-0.393701,0.865455,False,0,False,0
2,2026-07-15,Abaala,Garlic,Garlic (KG),Afar,2026,7,2.49,2.38,2.54,...,2.787143,0.089629,0.071740,0.701040,-5.928854,0.905455,False,0,False,0
3,2024-05-15,Abaala,Lentils,Lentils (100 KG),Afar,2024,5,349.53,182.08,175.29,...,89.673333,101.984827,96.734333,96.734333,8408.411215,1.748970,False,0,False,0
4,2024-06-15,Abaala,Lentils,Lentils (100 KG),Afar,2024,6,209.33,349.53,2.14,...,126.795714,173.732418,131.409554,132.077405,91.965070,1.047441,False,0,False,0


In [2]:
# We build the feature matrix using the engineered variables that represent recent price history,
# volatility, and how far the current price sits from its normal range.
# These are the signals most likely to separate true spikes from regular price fluctuations.
feature_cols = [
    'year', 'month_of_year', 'lag_1', 'lag_3', 'lag_6',
    'rolling_mean_3', 'rolling_mean_6', 'rolling_mean_12',
    'rolling_std_3', 'rolling_std_6', 'rolling_std_12',
    'pct_change_1m', 'price_to_mean_ratio', 'price_outlier_flag'
]
X = feature_df[feature_cols]
y = feature_df['is_spike_target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')
print(f'Positive rate in train: {y_train.mean():.4f}')

Train shape: (30723, 14), Test shape: (10242, 14)
Positive rate in train: 0.0367


In [3]:
# We compare several classifiers because price spikes are rare and can be hard to detect.
# Logistic regression gives a simple baseline; random forest captures nonlinear patterns;
# XGBoost is often strongest for tabular classification with class imbalance.
# We prioritize F1-score and ROC-AUC because the dataset is imbalanced and accuracy alone can be misleading.
# For final production work, we also select the probability threshold that maximizes F1 instead of using the default 0.5 cutoff.
models = {
    'logistic_regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'random_forest': RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42),
    'xgboost': XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42, scale_pos_weight=4, objective='binary:logistic', eval_metric='logloss')
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]

    best_threshold = 0.5
    best_f1 = -1.0
    best_precision = 0.0
    best_recall = 0.0
    best_confusion = None

    for threshold in np.linspace(0.10, 0.90, 81):
        pred = (prob >= threshold).astype(int)
        precision = precision_score(y_test, pred, zero_division=0)
        recall = recall_score(y_test, pred, zero_division=0)
        f1 = f1_score(y_test, pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
            best_precision = precision
            best_recall = recall
            best_confusion = confusion_matrix(y_test, pred)

    tn, fp, fn, tp = best_confusion.ravel()
    roc_auc = roc_auc_score(y_test, prob)
    results.append({
        'model': name,
        'precision': best_precision,
        'recall': best_recall,
        'f1': best_f1,
        'roc_auc': roc_auc,
        'best_threshold': best_threshold,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'tn': tn
    })
    print(f'{name.upper()} -> Threshold: {best_threshold:.2f}, Precision: {best_precision:.3f}, Recall: {best_recall:.3f}, F1: {best_f1:.3f}, ROC-AUC: {roc_auc:.3f}')

metrics_df = pd.DataFrame(results).sort_values('f1', ascending=False).reset_index(drop=True)
metrics_df.to_csv(root / 'models' / 'spike_metrics.csv', index=False)
print('\nTop spike classification model:')
print(metrics_df.head())


c:\Users\hp\OneDrive\Desktop\projecttt\ethiopian-food-price-watch\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LOGISTIC_REGRESSION -> Precision: 0.076, Recall: 0.662, F1: 0.137, ROC-AUC: 0.753
RANDOM_FOREST -> Precision: 0.786, Recall: 0.205, F1: 0.325, ROC-AUC: 0.880
XGBOOST -> Precision: 0.571, Recall: 0.269, F1: 0.365, ROC-AUC: 0.877

Top spike classification model:
                 model  precision    recall        f1   roc_auc   tp    fp  \
0              xgboost   0.570621  0.268617  0.365280  0.877322  101    76   
1        random_forest   0.785714  0.204787  0.324895  0.879988   77    21   
2  logistic_regression   0.076380  0.662234  0.136964  0.752843  249  3011   

    fn    tn  
0  275  9790  
1  299  9845  
2  127  6855  


In [4]:
# Save the selected model and the threshold value that maximizes F1 for deployment decisions.
import json

best_model_name = metrics_df.iloc[0]['model']
best_model = models[best_model_name]
best_threshold = float(metrics_df.iloc[0]['best_threshold'])

joblib.dump(best_model, root / 'models' / 'spike_classifier_model.joblib')
with open(root / 'models' / 'spike_threshold.json', 'w') as f:
    json.dump({'threshold': best_threshold, 'model': best_model_name}, f)

print(f'BEST MODEL SAVED: {best_model_name}')
print(f'BEST THRESHOLD SAVED: {best_threshold:.2f}')


BEST MODEL SAVED: xgboost
